In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
pd.set_option("display.max_columns", None)
folder = Path(r"C:\Users\Nonso_Fidelis\Desktop\Nonso Fidelis 2026\stocks-data-main\cache\nifty50_daily")

dfs = []

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    df["Symbol"] = file.stem.replace("_daily", "")
    dfs.append(df)

nifty = pd.concat(dfs, ignore_index=True)

In [3]:
nifty.head(7)

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,"ï»¿""Symbol""",PrevClose,OpenPrice,HighPrice,LowPrice,LastPrice,ClosePrice,AveragePrice,TotalTradedQuantity,TurnoverInRs,No.ofTrades,DeliverableQty,%DlyQttoTradedQty,Symbol
0,2002-07-01,-0.012187,-0.012522,-0.011920,-0.012173,1080397.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ADANIENT
1,2002-07-02,-0.012385,-0.012426,-0.012118,-0.012269,1016147.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ADANIENT
2,2002-07-03,-0.012255,-0.012392,-0.012194,-0.012269,980394.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ADANIENT
3,2002-07-04,-0.012324,-0.012522,-0.012324,-0.012337,972747.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ADANIENT
4,2002-07-05,-0.012406,-0.012406,-0.012262,-0.012310,974496.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ADANIENT
5,2002-07-08,-0.012461,-0.012995,-0.012194,-0.012851,1061686.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ADANIENT
6,2002-07-09,-0.012803,-0.012871,-0.012433,-0.012515,1024719.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ADANIENT


#### From our exploration, i ahve observed that the NIFTY50 dataset is different from the constituent data, so the machine learning problem should also be different. Also from exploration, two datasets were extracted from NIFTY50, namely:

YAHOO - which is Historical daily prices (Open, High, Low, Close, Volume) for all NIFTY50 stocks over many years.

NSE -  which is Daily NSE trading activity (No. of Trades, Deliverable Quantity, Turnover, etc.), available for a more recent period.

#### I will therefore be using the yahoo dataset for the ML model because it has over 300,000 observations, consistent columns across all stocks, and is better suited for prediction than the smaller nse dataset.

In [5]:
yahoo = (nifty.loc[nifty["Open"].notna(),["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"]].copy())

In [6]:
yahoo["Date"] = pd.to_datetime(yahoo["Date"])
yahoo = yahoo.sort_values(["Symbol", "Date"])

yahoo["DailyReturn"] = yahoo.groupby("Symbol")["Close"].pct_change()
yahoo["PriceRange"] = yahoo["High"] - yahoo["Low"]

yahoo["MA5"] = yahoo.groupby("Symbol")["Close"].transform(lambda x: x.rolling(5).mean())
yahoo["MA20"] = yahoo.groupby("Symbol")["Close"].transform(lambda x: x.rolling(20).mean())

yahoo["RollingVolatility"] = (yahoo.groupby("Symbol")["DailyReturn"].transform(lambda x: x.rolling(20).std()))

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


yahoo["TomorrowVolume"] = yahoo.groupby("Symbol")["Volume"].shift(-1) # This is to create target variable
yahoo = yahoo.dropna().reset_index(drop=True)
features = ["Open", "High", "Low", "Close", "Volume","DailyReturn", "PriceRange", "MA5","MA20", "RollingVolatility"]

X = yahoo[features]
y = yahoo["TomorrowVolume"]

### Data Splitting, Scaling and Training the models on the data (Linear Regression and DecisionTreeRegressor)

#### Note: I am treating this task as a regression task because the variable we are trying to predict is continuous rather than categorical.

### The objective of this regression task is to predict the next day's trading volume for each stock using historical market features such as prices, trading volume, daily returns, moving averages, price range, and rolling volatility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression().fit(X_train_scaled, y_train)
dt = DecisionTreeRegressor(max_depth=10, random_state=42).fit(X_train, y_train)

In [13]:
lr_pred = lr.predict(X_test_scaled)
dt_pred = dt.predict(X_test)

# Model comparison
results = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree"],
    "MAE": [
        mean_absolute_error(y_test, lr_pred),
        mean_absolute_error(y_test, dt_pred)
    ],
    "RMSE": [
        mean_squared_error(y_test, lr_pred) ** 0.5,
        mean_squared_error(y_test, dt_pred) ** 0.5
    ],
    "R²": [
        r2_score(y_test, lr_pred),
        r2_score(y_test, dt_pred)
    ]
})

print(results.round(3))

               Model          MAE          RMSE     R²
0  Linear Regression  4153109.074  1.494029e+07  0.728
1      Decision Tree  4080214.372  1.743334e+07  0.630


### The Linear Regression model achieved the best overall performance with an R² of 0.728, meaning it explains approximately 72.8% of the variation in the next day's trading volume. Although the Decision Tree produced a slightly lower MAE (4.08 million vs. 4.15 million), it had a much higher RMSE and a lower R² (0.630), indicating it made larger prediction errors on some observations. Overall, Linear Regression provides more reliable and consistent predictions, suggesting that the relationship between the selected market indicators and future trading volume is largely linear.

## Feature importance

In [14]:

importance = pd.DataFrame({"Feature": features,"Importance": dt.feature_importances_}).sort_values("Importance", ascending=False)
importance

,Feature,Importance
4,Volume,0.894243
6,PriceRange,0.022017
5,DailyReturn,0.021595
9,RollingVolatility,0.018648
0,Open,0.012012
8,MA20,0.010967
1,High,0.006285
7,MA5,0.005194
3,Close,0.004586
2,Low,0.004453


### The feature importance results show that today's trading volume is by far the strongest predictor of tomorrow's trading volume, contributing 89.4% of the model's predictive power. This suggests that trading activity tends to persist from one trading day to the next. PriceRange, DailyReturn, and RollingVolatility have relatively small but meaningful contributions, indicating that larger price movements and higher market volatility are associated with changes in future trading activity. In contrast, individual price variables such as Open, High, Low, and Close, along with the moving averages (MA5 and MA20), have minimal influence on predicting the next day's trading volume, implying that recent trading activity is a much stronger indicator than price levels alone.